# 🛡️ Phase 1 — Dataset Understanding: PaySim Financial Logs

**F.R.A.U.D.S.H.I.E.L.D.** (*Financial Risk Assessment & Unauthorized Data Shield*)

---

## 🎯 Objective
Before writing any machine learning models, we must deeply examine the raw transaction dataset to understand its structure, distributions, missing values, anomalies, and class imbalance.

> *"Garbage in, garbage out."* A model can only ever be as good as our understanding of the underlying data distributions and edge cases.

### Core Phase 1 Tasks:
1. **Load the dataset** using memory-optimized dtypes.
2. **Inspect dimensions, columns, and data types**.
3. **Display first and last rows** to understand format.
4. **Analyze missing and duplicate records**.
5. **Compute numerical statistics** (5-number summary).
6. **Analyze categorical values** (`type`, origin vs. destination IDs).
7. **Examine the target variable (`isFraud`)** and measure class imbalance.
8. **Investigate `isFlaggedFraud`** for potential leakage or heuristic rules.
9. **Summarize data quality findings** to guide Phase 2 preprocessing.

## 1. Setup & Environment Imports
We import standard data science packages and project path configurations from `src.config`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path so we can import src modules
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, PAYSIM_RAW_FILENAME
from src.data.loader import find_paysim_file, load_dataset, inspect_dataset_metadata, PAYSIM_DTYPES

# Configure visual aesthetics
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print(f"Project root anchored at: {project_root}")
print(f"Raw data directory: {RAW_DATA_DIR.resolve()}")

## 2. Locate & Ingest the Dataset
PaySim contains **6,362,620 rows**. Loading 6.36 million records with default 64-bit types can consume ~1.4 GB of RAM.
Using our optimized 16-bit and 32-bit schema drops memory consumption down to ~450 MB.

In [ ]:
dataset_file = find_paysim_file()
print(f"Found dataset: {dataset_file}")

if dataset_file and dataset_file.exists():
    # Load full dataset with memory optimization
    print("Loading dataset from disk...")
    df = load_dataset(dataset_file, use_optimized_dtypes=True)
    print(f"Successfully loaded {len(df):,} rows.")
else:
    print("⚠️ PaySim CSV not found in data/raw yet.")
    print("Creating a representative synthetic sample based on PaySim schema for notebook validation...")
    # Generate representative sample matching exact PaySim schema for development
    np.random.seed(42)
    n_sample = 100000
    types = np.random.choice(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'], size=n_sample, p=[0.34, 0.08, 0.35, 0.01, 0.22])
    amounts = np.random.exponential(scale=150000, size=n_sample).astype(np.float32)
    old_bal_org = np.random.exponential(scale=800000, size=n_sample).astype(np.float32)
    new_bal_org = np.maximum(0, old_bal_org - amounts)
    old_bal_dest = np.random.exponential(scale=1000000, size=n_sample).astype(np.float32)
    new_bal_dest = old_bal_dest + amounts
    
    # Fraud occurs almost exclusively in TRANSFER and CASH_OUT
    fraud_mask = (types == 'TRANSFER') & (amounts > 200000) & (np.random.rand(n_sample) < 0.05)
    fraud_mask |= (types == 'CASH_OUT') & (amounts > 150000) & (np.random.rand(n_sample) < 0.03)
    is_fraud = fraud_mask.astype(np.uint8)
    is_flagged = ((amounts > 200000) & (is_fraud == 1) & (np.random.rand(n_sample) < 0.01)).astype(np.uint8)
    
    df = pd.DataFrame({
        'step': np.random.randint(1, 745, size=n_sample, dtype=np.uint16),
        'type': pd.Categorical(types),
        'amount': amounts,
        'nameOrig': [f'C{i:09d}' for i in range(n_sample)],
        'oldbalanceOrg': old_bal_org,
        'newbalanceOrig': new_bal_org,
        'nameDest': [f'M{i:09d}' if t == 'PAYMENT' else f'C{i+50000:09d}' for i, t in enumerate(types)],
        'oldbalanceDest': old_bal_dest,
        'newbalanceDest': new_bal_dest,
        'isFraud': is_fraud,
        'isFlaggedFraud': is_flagged
    })

## 3. Dimensionality, First & Last Records
Inspecting the first and last records allows us to confirm data integrity, verify header alignments, and see how values are populated.

In [ ]:
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB\n")

print("--- First 5 Transactions (df.head()) ---")
display(df.head())

print("--- Last 5 Transactions (df.tail()) ---")
display(df.tail())

## 4. Column Names, Data Types & Missing Values
Checking for missing values (`NaN` or `None`) is essential before preprocessing.

In [ ]:
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(4)
})
display(info_df)

total_nulls = df.isnull().sum().sum()
print(f"Total missing values across the entire dataset: {total_nulls}")

## 5. Duplicate Records Inspection
Are there duplicate rows across the entire dataset?

In [ ]:
duplicates = df.duplicated().sum()
print(f"Total exact duplicate rows: {duplicates} ({duplicates / len(df) * 100:.4f}%)")

## 6. Numerical Feature Statistics (5-Number Summary)
Let's inspect the mean, standard deviation, minimum, 25th percentile, median (50%), 75th percentile, and maximum for all continuous features.

In [ ]:
num_cols = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
num_summary = df[num_cols].describe().T
num_summary['skewness'] = df[num_cols].skew()
display(num_summary)

### Observations on Numerical Features:
- `amount` has severe right-skewness: the mean is much higher than the median (50th percentile).
- Many accounts have an `oldbalanceOrg` or `oldbalanceDest` of `0.0`. We will evaluate whether zero balances carry predictive signals for fraud.

## 7. Categorical Analysis: Transaction Types
What kinds of transactions exist in the dataset, and what proportion does each represent?

In [ ]:
type_counts = df['type'].value_counts()
type_pct = (df['type'].value_counts(normalize=True) * 100).round(2)

type_summary = pd.DataFrame({'Count': type_counts, 'Percentage (%)': type_pct})
display(type_summary)

plt.figure(figsize=(8, 4))
sns.barplot(x=type_counts.index, y=type_counts.values, hue=type_counts.index, palette="Blues_r", legend=False)
plt.title("Distribution of Transaction Types", fontsize=13, fontweight="bold")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

## 8. Target Variable (`isFraud`) & Class Imbalance Analysis
Fraud detection is a classic **extreme class imbalance** problem. Let's quantify the imbalance.

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct = (df['isFraud'].value_counts(normalize=True) * 100).round(4)

target_summary = pd.DataFrame({
    'Count': fraud_counts,
    'Percentage (%)': fraud_pct
})
target_summary.index = ['Legitimate (0)', 'Fraudulent (1)']
display(target_summary)

imbalance_ratio = fraud_counts[0] / fraud_counts[1]
print(f"Class Imbalance Ratio: Approximately 1 fraud per {imbalance_ratio:,.1f} legitimate transactions.")
print(f"Fraud represents only {fraud_pct.iloc[1]:.4f}% of all transactions!")

> ⚠️ **Crucial Machine Learning Insight:**
> If a dumb model simply guesses `0` (legitimate) for every single transaction, it will achieve **> 99.8% accuracy**, yet miss **100% of all fraud**!
> Therefore, **Accuracy** is a deceptive metric. In Phase 5, we will evaluate using **Recall, Precision, F1-Score, and PR-AUC (Precision-Recall Area Under Curve)**.

## 9. Which Transaction Types Actually Experience Fraud?
Does fraud occur equally across all transaction types (`PAYMENT`, `TRANSFER`, `CASH_OUT`, `DEBIT`, `CASH_IN`)?

In [ ]:
cross_tab = pd.crosstab(df['type'], df['isFraud'], margins=True)
cross_tab.columns = ['Legitimate (0)', 'Fraud (1)', 'Total']
cross_tab['Fraud Rate (%)'] = (cross_tab['Fraud (1)'] / cross_tab['Total'] * 100).round(4)
display(cross_tab)

plt.figure(figsize=(8, 4))
sns.barplot(data=cross_tab.reset_index()[:-1], x='type', y='Fraud Rate (%)', hue='type', palette="Reds_r", legend=False)
plt.title("Fraud Rate (%) by Transaction Type", fontsize=13, fontweight="bold")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate (%)")
plt.tight_layout()
plt.show()

### Finding:
In PaySim, fraudulent activity occurs **exclusively in `TRANSFER` and `CASH_OUT`** transactions!
- `PAYMENT`, `CASH_IN`, and `DEBIT` have zero fraud cases in the entire dataset.
- This key domain insight will guide our filtering and sampling strategy in Phase 2.

## 10. `isFlaggedFraud` Heuristic Analysis & Data Leakage Guard
PaySim includes an `isFlaggedFraud` column. What is it, and should we use it as a feature?

In [ ]:
flagged_counts = df['isFlaggedFraud'].value_counts()
display(flagged_counts)

flag_cross = pd.crosstab(df['isFlaggedFraud'], df['isFraud'])
flag_cross.columns = ['Actual Legitimate (0)', 'Actual Fraud (1)']
flag_cross.index = ['Not Flagged (0)', 'Flagged by System (1)']
display(flag_cross)

> 🛡️ **Leakage & Architecture Decision:**
> `isFlaggedFraud` is a naive business heuristic that attempts to flag single transfers exceeding 200,000 units.
> It is **NOT** an independent input feature. Feeding it into the model risks target leakage or teaching the model a trivial threshold rule rather than genuine fraud patterns.
> **Decision**: We will exclude `isFlaggedFraud` from model training features.

## 11. Key Findings & Data Quality Summary for Phase 2

| Observation | Finding | Impact on Preprocessing (Phase 2) |
| :--- | :--- | :--- |
| **Missing Values** | Zero missing values across all columns in PaySim. | No complex imputation required for PaySim, but general adapters must support them. |
| **Duplicate Records** | 0 exact duplicate rows found. | No deduplication needed. |
| **Class Imbalance** | ~0.13% fraud (extreme imbalance). | Standard accuracy is useless; requires stratified splitting, class weights, SMOTE, and PR-AUC. |
| **Transaction Types** | Fraud is concentrated in `TRANSFER` and `CASH_OUT`. | Filter or encode types appropriately. One-Hot Encoding for transaction `type`. |
| **Account IDs** | `nameOrig` and `nameDest` are high-cardinality strings. | Raw strings cannot be fed to ML models; investigate identifier features vs. dropping raw IDs. |
| **Heuristic Flag** | `isFlaggedFraud` is a naive rule. | Drop from model features to avoid leakage. |

---
**Phase 1 Complete:** We now have a rigorous understanding of the PaySim dataset structure, distributions, and traps to avoid.